In [113]:
import pandas as pd

df = pd.read_excel("../data/train.xlsx")

df

,review_description,rating
0,شركه زباله و سواقين بتبرشم و مفيش حتي رقم للشك...,-1
1,خدمة الدفع عن طريق الكي نت توقفت عندي اصبح فقط...,1
2,تطبيق غبي و جاري حذفه ، عاملين اكواد خصم و لما...,-1
3,فعلا تطبيق ممتاز بس لو فى امكانية يتيح لمستخدم...,1
4,سيء جدا ، اسعار رسوم التوصيل لا تمت للواقع ب ص...,-1
...,...,...
32031,التطبيق اصبح سيء للغايه نقوم بطلب لا يتم وصول ...,-1
32032,y love you,1
32033,الباقه بتخلص وبشحن مرتين باقه اضافيه ١٠٠ جنيه,-1
32034,تطبيق فاشل وصلني الطلب ناقص ومش ينفع اعمل حاجة...,-1


In [114]:
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download("stopwords")
nltk.download("punkt")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mereh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mereh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [115]:
df = df.dropna()

df = df.drop_duplicates()

df["review_description"] = df["review_description"].astype(str)

In [116]:
arabic_stopwords = set(stopwords.words("arabic"))

negation_words = {
    "لا", "لم", "لن", "ليس", "ما", "غير",
    "بدون", "دون", "ولا", "ليسوا", "ليست"
}

arabic_stopwords = arabic_stopwords - negation_words

In [117]:
emoji_df = pd.read_csv("../data/emoji.csv")


In [118]:
emoji_df

,ID,Emoji,Unicode,Sentiment,Description_Arabic
0,1,😊,U+1F60A,Positive,وجه مبتسم بعيون مبتسمة، يعبر عن السعادة العامة...
1,2,❤️,U+2764,Positive,قلب أحمر، الرمز العالمي للحب والمودة والعاطفة ...
2,3,😂,U+1F602,Positive,وجه يضحك مع دموع الفرح، يستخدم للتعبير عن ضحك ...
3,4,👍,U+1F44D,Positive,علامة الإبهام لأعلى، تدل على الموافقة، الإعجاب...
4,5,😭,U+1F62D,Negative,وجه يبكي بحرقة، يعبر عن الحزن الشديد، الألم، ا...
...,...,...,...,...,...
1541,1542,⚜️,U+269C,Neutral,زهرة الزنبق (Fleur-de-lis)، رمز فرنسي أو ملكي.
1542,1543,🔱,U+1F531,Neutral,رمح ثلاثي الشعب (ترايدنت)، يرمز لبوسيدون/نبتون...
1543,1544,📛,U+1F4DB,Neutral,شارة اسم (يابانية).
1544,1545,🔰,U+1F530,Neutral,علامة سائق جديد يابانية (شودنشا).


In [119]:
emoji_dict = dict(
    zip(
        emoji_df["Emoji"],
        emoji_df["Description_Arabic"]
    )
)


In [120]:
emoji_items = sorted(
    emoji_dict.items(),
    key=lambda x: len(x[0]),
    reverse=True
)

def replace_emojis(text):
    for emoji_char, description in emoji_items:
        text = text.replace(
            emoji_char,
            " " + description + " "
        )
    return text

In [121]:
import re
from nltk.tokenize import word_tokenize

# Function to preprocess Arabic text
def preprocess(text):

    # ==========================
    # Remove URLs
    # ==========================
    text = re.sub(r"http\S+|www\S+", "", text)

    # ==========================
    # Remove HTML Tags
    # ==========================
    text = re.sub(r"<.*?>", "", text)

    # ==========================
    # Remove Emojis
    # ==========================
    text = replace_emojis(text)   

    # ==========================
    # Remove Mentions (@username)
    # and Hashtags (#topic)
    # ==========================
    text = re.sub(r"[@#]\w+", "", text)

    # ==========================
    # Remove English Letters
    # ==========================
    text = re.sub(r"[A-Za-z]+", "", text)

    # ==========================
    # Remove Numbers
    # (Arabic & English)
    # ==========================
    text = re.sub(r"[0-9٠-٩]", "", text)

    # ==========================
    # Remove Punctuation
    # ==========================
    text = re.sub(r"[^\w\s]", " ", text)

    # ==========================
    # Remove Arabic Diacritics
    # ==========================
    text = re.sub(r"[ًٌٍَُِّْـ]", "", text)

    # ==========================
    # Normalize Arabic Letters
    # ==========================
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)

    # ==========================
    # Remove Repeated Characters
    # Example:
    # راااائع -> رائع
    # حلوووو -> حلو
    # ==========================
    text = re.sub(r"(.)\1{2,}", r"\1", text)

    # ==========================
    # Keep Arabic Letters Only
    # ==========================
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)

    # ==========================
    # Remove Extra Spaces
    # ==========================
    text = re.sub(r"\s+", " ", text).strip()

    # ==========================
    # Tokenization
    # ==========================
    words = word_tokenize(text)

    # ==========================
    # Remove Arabic Stopwords
    # ==========================
    words = [word for word in words if word not in arabic_stopwords]


    # ==========================
    # Join Tokens Again
    # ==========================
    return " ".join(words)

In [122]:
df["clean_text"] = df["review_description"].apply(preprocess)

df = df[df["clean_text"].str.strip() != ""]

df = df[df["clean_text"].str.split().str.len() >= 2]

df.reset_index(drop=True, inplace=True)

In [123]:

print(df["rating"].value_counts())

rating
 1    16028
-1    10472
 0     1372
Name: count, dtype: int64


In [124]:
df[["review_description", "clean_text", "rating"]].head(10)

,review_description,clean_text,rating
0,شركه زباله و سواقين بتبرشم و مفيش حتي رقم للشك...,شركه زباله سواقين بتبرشم مفيش حتي رقم للشكاوي ...,-1
1,خدمة الدفع عن طريق الكي نت توقفت عندي اصبح فقط...,خدمة الدفع طريق الكي نت توقفت عندي اصبح فقط ال...,1
2,تطبيق غبي و جاري حذفه ، عاملين اكواد خصم و لما...,تطبيق غبي جاري حذفه عاملين اكواد خصم نستخدمها ...,-1
3,فعلا تطبيق ممتاز بس لو فى امكانية يتيح لمستخدم...,فعلا تطبيق ممتاز امكانية يتيح لمستخدم التطبيق ...,1
4,سيء جدا ، اسعار رسوم التوصيل لا تمت للواقع ب ص...,سيء جدا اسعار رسوم التوصيل لا تمت للواقع صله,-1
5,قعد عشرين سنة يدور على سائق بس اما عن توصيل ال...,قعد سنة يدور علي سايق اما توصيل الاشياء جيد جدا,0
6,احلئ تطبيق,احلي تطبيق,1
7,رائع واو مدهش,رايع مدهش,1
8,مکو بس البحرین وعمان وغیرهه بس العراق مکو یعنی...,مکو البحرین وعمان وغیرهه العراق مکو یعنی نجمه ...,-1
9,تطبيق جميل يستاهل الخمس نجوم👍👍👍,تطبيق جميل يستاهل الخمس نجوم علامة الابهام لاع...,1


In [125]:
lengths = df["clean_text"].str.split().apply(len)

print(lengths.describe())

count    27872.000000
mean        14.909084
std         51.028548
min          2.000000
25%          3.000000
50%          6.000000
75%         15.000000
max       3723.000000
Name: clean_text, dtype: float64


In [126]:
from sklearn.model_selection import train_test_split

X = df["clean_text"]
y = df["rating"]

y = df["rating"].replace({
    -1: 0,
     0: 1,
     1: 2
})
x_train, x_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [127]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(zip(np.unique(y_train), class_weights))

print(class_weights)

{np.int64(0): np.float64(0.8872309100314353), np.int64(1): np.float64(6.768973891924712), np.int64(2): np.float64(0.5796547600478344)}


In [128]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

VOCAB_SIZE = 30000
MAX_LEN = 50

tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(x_train)

x_train = tokenizer.texts_to_sequences(x_train)
x_test = tokenizer.texts_to_sequences(x_test)

x_train = pad_sequences(
    x_train,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

x_test = pad_sequences(
    x_test,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

In [129]:
import time
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

lstm = Sequential()

# Embedding Layer
lstm.add(
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128,
        input_length=MAX_LEN
    )
)

# LSTM Layer
lstm.add(
    LSTM(
        64,
        dropout=0.3,
        recurrent_dropout=0.3
    )
)

# Hidden Layer
lstm.add(Dense(64, activation="relu"))

# Dropout Layer
lstm.add(Dropout(0.5))

# Output Layer
lstm.add(Dense(3, activation="softmax"))

optimizer = Adam(learning_rate=0.0005)

lstm.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

start_time = time.time()

early_stop_lstm = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history_lstm = lstm.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=64,
    callbacks=[early_stop_lstm],
    class_weight=class_weights
)

end_time = time.time()

training_time_lstm = end_time - start_time

print(f"Training Time: {training_time_lstm:.2f} seconds")
print(f"Training Time: {training_time_lstm/60:.2f} minutes")

Epoch 1/20


c:\Users\mereh\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


279/279 ━━━━━━━━━━━━━━━━━━━━ 16s 48ms/step - accuracy: 0.2802 - loss: 1.1029 - val_accuracy: 0.3955 - val_loss: 1.0814
Epoch 2/20
279/279 ━━━━━━━━━━━━━━━━━━━━ 13s 46ms/step - accuracy: 0.3271 - loss: 1.0935 - val_accuracy: 0.4724 - val_loss: 1.0313
Epoch 3/20
279/279 ━━━━━━━━━━━━━━━━━━━━ 13s 46ms/step - accuracy: 0.4290 - loss: 1.0424 - val_accuracy: 0.7724 - val_loss: 0.8757
Epoch 4/20
279/279 ━━━━━━━━━━━━━━━━━━━━ 13s 46ms/step - accuracy: 0.7076 - loss: 0.9994 - val_accuracy: 0.6868 - val_loss: 1.1631
Epoch 5/20
279/279 ━━━━━━━━━━━━━━━━━━━━ 13s 46ms/step - accuracy: 0.5530 - loss: 1.0190 - val_accuracy: 0.5312 - val_loss: 1.0318
Training Time: 67.72 seconds
Training Time: 1.13 minutes


In [130]:
import time
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

gru = Sequential()

# Embedding Layer
gru.add(
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=128,
        input_length=MAX_LEN
    )
)

# GRU Layer
gru.add(
    GRU(
        64,
        dropout=0.3,
        recurrent_dropout=0.3
    )
)

# Hidden Layer
gru.add(Dense(64, activation="relu"))

# Dropout Layer
gru.add(Dropout(0.5))

# Output Layer
gru.add(Dense(3, activation="softmax"))

optimizer = Adam(learning_rate=0.0005)

gru.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

start_time = time.time()

early_stop_gru = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history_gru = gru.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=64,
    callbacks=[early_stop_gru],
    class_weight=class_weights
)

end_time = time.time()

training_time_gru = end_time - start_time

print(f"Training Time: {training_time_gru:.2f} seconds")
print(f"Training Time: {training_time_gru/60:.2f} minutes")

Epoch 1/20
279/279 ━━━━━━━━━━━━━━━━━━━━ 16s 46ms/step - accuracy: 0.3516 - loss: 1.1036 - val_accuracy: 0.5807 - val_loss: 1.0901
Epoch 2/20
279/279 ━━━━━━━━━━━━━━━━━━━━ 13s 45ms/step - accuracy: 0.2746 - loss: 1.0990 - val_accuracy: 0.0816 - val_loss: 1.0967
Epoch 3/20
279/279 ━━━━━━━━━━━━━━━━━━━━ 13s 45ms/step - accuracy: 0.2653 - loss: 1.0911 - val_accuracy: 0.4054 - val_loss: 1.0866
Epoch 4/20
279/279 ━━━━━━━━━━━━━━━━━━━━ 12s 44ms/step - accuracy: 0.3245 - loss: 1.0755 - val_accuracy: 0.1417 - val_loss: 1.0958
Epoch 5/20
279/279 ━━━━━━━━━━━━━━━━━━━━ 13s 47ms/step - accuracy: 0.4594 - loss: 1.0198 - val_accuracy: 0.7877 - val_loss: 0.8175
Epoch 6/20
279/279 ━━━━━━━━━━━━━━━━━━━━ 13s 46ms/step - accuracy: 0.7514 - loss: 0.9017 - val_accuracy: 0.8002 - val_loss: 0.7488
Epoch 7/20
279/279 ━━━━━━━━━━━━━━━━━━━━ 13s 46ms/step - accuracy: 0.7782 - loss: 0.7805 - val_accuracy: 0.6966 - val_loss: 0.6721
Epoch 8/20
279/279 ━━━━━━━━━━━━━━━━━━━━ 13s 46ms/step - accuracy: 0.7883 - loss: 0.6667 - 

In [131]:
lstm.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_8 (Embedding)         │ (None, 50, 128)        │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,681,291 (44.56 MB)

 Trainable params: 3,893,763 (14.85 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 7,787,528 (29.71 MB)

In [132]:
gru.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)         │ (None, 50, 128)        │     3,840,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_4 (GRU)                     │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,644,811 (44.42 MB)

 Trainable params: 3,881,603 (14.81 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 7,763,208 (29.61 MB)

In [133]:
loss_lstm, accuracy_lstm = lstm.evaluate(
    x_test,
    y_test
)
print("Loss:", loss_lstm)
print("Accuracy:", accuracy_lstm)

175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7648 - loss: 0.8820
Loss: 0.8819924592971802
Accuracy: 0.7648430466651917


In [134]:
loss_gru, accuracy_gru = gru.evaluate(x_test, y_test)

print("Loss:", loss_gru)
print("Accuracy:", accuracy_gru)

175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7396 - loss: 0.7000
Loss: 0.6999859809875488
Accuracy: 0.7395515441894531


In [135]:
from sklearn.metrics import classification_report
import numpy as np

y_pred_lstm = lstm.predict(x_test)

y_pred_lstm = np.argmax(y_pred_lstm, axis=1)

print("LSTM Classification Report")
print(
    classification_report(
        y_test,
        y_pred_lstm,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ]
    )
)

175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step
LSTM Classification Report
              precision    recall  f1-score   support

    Negative       0.78      0.63      0.70      2095
     Neutral       0.00      0.00      0.00       274
    Positive       0.76      0.92      0.83      3206

    accuracy                           0.76      5575
   macro avg       0.51      0.52      0.51      5575
weighted avg       0.73      0.76      0.74      5575



c:\Users\mereh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\mereh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\mereh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [136]:
from sklearn.metrics import classification_report
import numpy as np

y_pred_gru = gru.predict(x_test)

y_pred_gru = np.argmax(y_pred_gru, axis=1)

print("GRU Classification Report")
print(
    classification_report(
        y_test,
        y_pred_gru,
        target_names=[
            "Negative",
            "Neutral",
            "Positive"
        ]
    )
)

175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
GRU Classification Report
              precision    recall  f1-score   support

    Negative       0.86      0.66      0.75      2095
     Neutral       0.11      0.43      0.18       274
    Positive       0.90      0.81      0.86      3206

    accuracy                           0.74      5575
   macro avg       0.63      0.64      0.59      5575
weighted avg       0.85      0.74      0.78      5575



In [137]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# LSTM Metrics
lstm_accuracy = accuracy_score(y_test, y_pred_lstm)
lstm_precision = precision_score(y_test, y_pred_lstm, average="weighted")
lstm_recall = recall_score(y_test, y_pred_lstm, average="weighted")
lstm_f1 = f1_score(y_test, y_pred_lstm, average="weighted")

# GRU Metrics
gru_accuracy = accuracy_score(y_test, y_pred_gru)
gru_precision = precision_score(y_test, y_pred_gru, average="weighted")
gru_recall = recall_score(y_test, y_pred_gru, average="weighted")
gru_f1 = f1_score(y_test, y_pred_gru, average="weighted")

comparison = pd.DataFrame({
    "Model": ["LSTM", "GRU"],
    "Accuracy": [lstm_accuracy, gru_accuracy],
    "Loss": [loss_lstm, loss_gru],
    "Precision": [lstm_precision, gru_precision],
    "Recall": [lstm_recall, gru_recall],
    "F1-Score": [lstm_f1, gru_f1],
    "Training Time (sec)": [
        training_time_lstm,
        training_time_gru
    ]

})

comparison

c:\Users\mereh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


,Model,Accuracy,Loss,Precision,Recall,F1-Score,Training Time (sec)
0,LSTM,0.764843,0.881992,0.729664,0.764843,0.739625,67.724519
1,GRU,0.739552,0.699986,0.848144,0.739552,0.783060,130.166981
